# Multilevel and Marginal Logistic Regression: A Case Study with the HSB Dataset

This notebook is a companion to the previous case study on linear models for dependent data. Here, we will focus on applying similar advanced regression techniques to a **binary outcome variable**. We will again use the "High School and Beyond" (HSB) dataset to explore marginal logistic regression using Generalized Estimating Equations (GEE).

We begin by importing the necessary libraries.

```python
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import statsmodels.api as sm
import numpy as np
```

### Data Preparation

First, we load the HSB dataset and prepare our variables. Our outcome of interest is `honors`, which indicates whether a student was enrolled in an honors program. This is a binary variable that we will code as `1` for "enrolled" and `0` for "not enrolled". Our predictors will be the student's math achievement score (`math_ach`) and gender (`female`).

The data has a natural multilevel structure, with students (Level 1) nested within schools (Level 2), identified by `school_id`.

```python
# Load the High School and Beyond (HSB) dataset
try:
    da = pd.read_csv("https://stats.idre.ucla.edu/stat/data/hsbdemo.csv")
except Exception as e:
    print(f"Failed to load data. Error: {e}")
    print("Please check your internet connection or the URL.")
    # As a fallback, create a dummy dataframe to avoid further errors
    da = pd.DataFrame({'id': [], 'female': [], 'ses': [], 'schtyp': [], 'prog': [], 'read': [], 'write': [], 'math': [], 'science': [], 'socst': [], 'honors': [], 'awards': [], 'cid': []})


# Rename columns and create our binary outcome and predictor variables
da = da.rename(columns={"id": "student_id", "math": "math_ach", "cid": "school_id"})
da["honors_enroll"] = da.honors.replace({"enrolled": 1, "not enrolled": 0})
da["is_female"] = da.female.replace({"female": 1, "male": 0})

# Select the variables we will use and drop any rows with missing data
vars_to_use = ["honors_enroll", "math_ach", "is_female", "school_id"]
da = da[vars_to_use].dropna()

# Display the first few rows
print(da.head())
```

```text
   honors_enroll  math_ach  is_female  school_id
0              0        41          0          1
1              0        53          1          1
2              0        54          0          1
3              0        47          0          1
4              0        57          0          1
```

## Marginal Logistic Regression with Dependent Data

Our goal is to model the probability of a student being in an honors program. Because students are clustered within schools, their outcomes are not independent. A standard logistic regression model (fit using `GLM`) would ignore this clustering and likely produce standard errors that are too small (i.e., overly confident).

We will compare a standard logistic `GLM` to a marginal logistic model fit using **Generalized Estimating Equations (GEE)**. The GEE model will account for the clustering and provide more robust and trustworthy inferences.

### Comparison: GLM (Ignoring Clustering) vs. GEE (Accounting for Clustering)

Both models will have the same **mean structure**: we are modeling the probability of `honors_enroll` as a function of `math_ach` and `is_female`.

The key difference is that the GEE model will also specify a **covariance structure** to account for the within-school correlation. We will use an `Exchangeable` structure, which assumes a constant correlation between any two students in the same school.

```python
# --- Model 1: Standard Logistic Regression (GLM) ---
# This model ignores the school-level clustering.
model1 = sm.GLM.from_formula("honors_enroll ~ math_ach + is_female",
           family=sm.families.Binomial(), data=da)
result1 = model1.fit()
print("--- Standard GLM Results (Ignoring Clustering) ---")
print(result1.summary())


# --- Model 2: Marginal Logistic Regression (GEE) ---
# This model accounts for clustering using 'groups="school_id"'.
model2 = sm.GEE.from_formula("honors_enroll ~ math_ach + is_female",
           groups="school_id", family=sm.families.Binomial(),
           cov_struct=sm.cov_struct.Exchangeable(), data=da)
result2 = model2.fit()
print("\n--- GEE Results (Accounting for Clustering) ---")
print(result2.summary())
```

```text
--- Standard GLM Results (Ignoring Clustering) ---
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:          honors_enroll   No. Observations:                  200
Model:                            GLM   Df Residuals:                      197
Model Family:                Binomial   Df Model:                            2
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -82.463
Date:                Tue, 21 Oct 2025   Deviance:                       164.93
Time:                        12:07:02   Pearson chi2:                      192.
No. Iterations:                     5   Pseudo R-squ. (CS):             0.1855
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -7.3948      1.309     -5.649      0.000     -9.960      -4.829
math_ach       0.1221      0.023      5.318      0.000      0.077       0.167
is_female     -0.3479      0.461     -0.755      0.450     -1.251       0.555
==============================================================================

--- GEE Results (Accounting for Clustering) ---
                          GEE Regression Results                          
================================================================================
Dep. Variable:                      honors_enroll   No. Observations:          200
Model:                                        GEE   No. clusters:               30
Model Family:                          Binomial   Min. cluster size:           1
Link Function:                            Logit   Max. cluster size:          14
Method:                        Iterative Fitting   Mean cluster size:         6.7
Time:                           Tue, 21 Oct 2025   Scale:                   1.000
Covariance Type:                        robust   Cov. Estimator:        Sandwich
================================================================================
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -7.3948      1.229     -6.016      0.000     -9.804      -4.986
math_ach         0.1221      0.021      5.794      0.000      0.081       0.163
is_female       -0.3479      0.428     -0.813      0.416     -1.187       0.491
================================================================================
```

### Comparing the Parameter Estimates and Standard Errors

Let's create a summary table to compare the key results directly.

```python
x = pd.DataFrame({"GLM_params": result1.params, "GLM_SE": result1.bse,
                  "GEE_params": result2.params, "GEE_SE": result2.bse})
x = x[["GLM_params", "GLM_SE", "GEE_params", "GEE_SE"]]
print(x)
```

```text
                   GLM_params    GLM_SE  GEE_params    GEE_SE
Intercept         -7.394843  1.308833   -7.394843  1.229227
math_ach           0.122110  0.022964    0.122110  0.021074
is_female         -0.347918  0.460803   -0.347918  0.427926
```

### Analysis of the Comparison

As expected, we see a pattern similar to the linear model case, but with some nuances:
1.  **Parameter Estimates:** The `GLM_params` and `GEE_params` are identical. For the population-average interpretation, both models arrive at the same best-fit estimates for the coefficients.
2.  **Standard Errors:** This is where the difference lies. In this particular run, the `GEE_SE` values are actually slightly *smaller* than the `GLM_SE` values. While often GEE standard errors are larger (to correct for the overly optimistic GLM), this is not guaranteed. The "sandwich" estimator used by GEE provides a *robust* estimate of the variance that correctly accounts for the clustering. The GLM standard error, in contrast, is simply incorrect because it is based on a faulty independence assumption. **The GEE standard errors are the ones we should trust for inference.**

### Interpretation of the GEE Model

Based on the GEE results, we can make the following population-average interpretations:

*   **`math_ach` (p < 0.001):** Math achievement is a highly significant predictor. To interpret the coefficient (`0.1221`), we calculate the odds ratio:
    *   Odds Ratio = $e^{0.1221} \approx 1.13$.
    *   **Interpretation:** "Averaged across all schools, for each one-point increase in a student's math score, their odds of being in an honors program are estimated to increase by about 13%."
*   **`is_female` (p = 0.416):** Gender is not a statistically significant predictor after accounting for math achievement.

## Summary and Conclusions

This case study demonstrates the importance and application of marginal models for binary dependent data.

1.  **Problem:** When data is clustered (e.g., students in schools), standard logistic regression (GLM) violates the independence assumption, leading to untrustworthy standard errors and inferences.
2.  **Solution:** Marginal models, fit using **GEE**, provide a robust solution. They estimate the same population-average effects but use a "sandwich" estimator to calculate standard errors that correctly account for the within-cluster dependency.
3.  **Benefits:** This approach gives us three key advantages over a naive GLM:
    *   It provides insight into the dependence structure of the data (via the estimated correlation parameter).
    *   It produces meaningful and robust standard errors, leading to valid confidence intervals and p-values.
    *   It can leverage the dependence structure to potentially produce more accurate parameter estimates.

For researchers whose primary interest is in population-average effects, GEE is an essential, powerful, and computationally efficient tool for modeling clustered binary data.